In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/GEE S2_Data/Traning/S2_49_TRAINING_data.csv")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/GEE S2_Data/Traning/S2_49_TRAINING_data.csv'

In [ ]:
count = len(df[df['LABEL'] == 1])
count

1500000

In [ ]:
len(df)

2560000

In [ ]:
count / len(df)

0.5859375

In [ ]:
!pip install -q lightgbm xgboost catboost pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 56.2 MB/s eta 0:00:00


In [ ]:
import os

print(os.listdir('/content/drive/MyDrive/GEE S2_Data/Traning'))

['S2_145_TRAINING_data.csv', 'RGB_Tiles', 'RGB_Cloud_Mask_Tiles', 'S2_149_TRAINING_data.csv', 'S2_157_TRAINING_data.csv', '.ipynb_checkpoints', 'S2_105_TRAINING_data.csv', 'S1_237_TRAINING_data.csv', 'S1_49_TRAINING_data.csv', 'S1_9_TRAINING_data.csv', 'S1_274_TRAINING_data.csv', 'S_117_TRAINING_data.csv']


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

FOLDER_PATH = '/content/drive/MyDrive/GEE S2_Data/Traning'
OUTPUT_PATH = '/content/drive/MyDrive/Models'

print("Files trong folder:")
print(os.listdir(FOLDER_PATH))

os.makedirs(OUTPUT_PATH, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files trong folder:
['S2_145_TRAINING_data.csv', 'RGB_Tiles', 'RGB_Cloud_Mask_Tiles', 'S2_149_TRAINING_data.csv', 'S2_157_TRAINING_data.csv', '.ipynb_checkpoints', 'S2_105_TRAINING_data.csv', 'S1_237_TRAINING_data.csv', 'S1_49_TRAINING_data.csv', 'S1_9_TRAINING_data.csv', 'S1_274_TRAINING_data.csv', 'S_117_TRAINING_data.csv']


In [ ]:
FOLDER_PATH = '/content/drive/MyDrive/GEE S2_Data/Training'

In [ ]:
csv_files = [
    os.path.join(FOLDER_PATH, f)
    for f in os.listdir(FOLDER_PATH)
    if f.endswith('.csv') and 'tile_info' not in f.lower()
]

print("Số file CSV:", len(csv_files))
print(csv_files)


Số file CSV: 9
['/content/drive/MyDrive/GEE S2_Data/Traning/S2_145_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S2_149_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S2_157_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S2_105_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S1_237_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S1_49_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S1_9_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S1_274_TRAINING_data.csv', '/content/drive/MyDrive/GEE S2_Data/Traning/S_117_TRAINING_data.csv']


In [ ]:
import pandas as pd
import numpy as np
import gc

FEATURES = [
    'B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B11','B12',
    'NDVI','NDSI','NDWI','B11_B08_Ratio','B02_B04_Ratio','NDMI',
    'B8A_B11_Ratio','B01_B11_Ratio','B05_B04_Ratio','B12_B03_Ratio',
    'Tile_Cloud_Coverage'
]
TARGET = 'LABEL'


In [ ]:
dfs = []

for i, file in enumerate(csv_files):
    print(f"Đọc {i+1}/{len(csv_files)}: {os.path.basename(file)}")

    df = pd.read_csv(
        file,
        usecols=lambda c: c in FEATURES + [TARGET, 'Tile_ID'],
        dtype=np.float32
    )

    dfs.append(df)
    del df
    gc.collect()

full_df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print("Dataset shape:", full_df.shape)


Đọc 1/9: S2_145_TRAINING_data.csv
Đọc 2/9: S2_149_TRAINING_data.csv
Đọc 3/9: S2_157_TRAINING_data.csv
Đọc 4/9: S2_105_TRAINING_data.csv
Đọc 5/9: S1_237_TRAINING_data.csv
Đọc 6/9: S1_49_TRAINING_data.csv
Đọc 7/9: S1_9_TRAINING_data.csv
Đọc 8/9: S1_274_TRAINING_data.csv
Đọc 9/9: S_117_TRAINING_data.csv
Dataset shape: (25437467, 25)


In [ ]:
import pandas as pd
import numpy as np
import random
import os
from time import time
import gc

CHUNK_SIZE = 500_000
MAX_PER_CLASS = 100_000

print("── PASS 1: TÌM INDEX ──")
start = time()

cloud_idx = []
clear_idx = []

global_index = 0

for f_id, file in enumerate(csv_files):
    print(f"[{f_id+1}/{len(csv_files)}] Scan LABEL:", os.path.basename(file))

    for chunk in pd.read_csv(
        file,
        usecols=[TARGET],
        chunksize=CHUNK_SIZE
    ):
        labels = chunk[TARGET].to_numpy()

        cloud_idx.extend(global_index + np.flatnonzero(labels == 1))
        clear_idx.extend(global_index + np.flatnonzero(labels == 0))

        global_index += len(chunk)

print("→ Tổng cloud:", len(cloud_idx))
print("→ Tổng clear:", len(clear_idx))


── PASS 1: TÌM INDEX ──
[1/9] Scan LABEL: S2_145_TRAINING_data.csv
[2/9] Scan LABEL: S2_149_TRAINING_data.csv
[3/9] Scan LABEL: S2_157_TRAINING_data.csv
[4/9] Scan LABEL: S2_105_TRAINING_data.csv
[5/9] Scan LABEL: S1_237_TRAINING_data.csv
[6/9] Scan LABEL: S1_49_TRAINING_data.csv
[7/9] Scan LABEL: S1_9_TRAINING_data.csv
[8/9] Scan LABEL: S1_274_TRAINING_data.csv
[9/9] Scan LABEL: S_117_TRAINING_data.csv
→ Tổng cloud: 14654119
→ Tổng clear: 10783348


In [ ]:
n_min = min(len(cloud_idx), len(clear_idx), MAX_PER_CLASS)

random.shuffle(cloud_idx)
random.shuffle(clear_idx)

final_indices = cloud_idx[:n_min] + clear_idx[:n_min]
random.shuffle(final_indices)

final_set = set(final_indices)

print("✔ Mỗi lớp:", n_min)
print("✔ Tổng mẫu:", len(final_indices))
print("⏱ Thời gian PASS 1:", time() - start)
print("──────────────────────────────\n")


✔ Mỗi lớp: 100000
✔ Tổng mẫu: 200000
⏱ Thời gian PASS 1: 27.137078285217285
──────────────────────────────



In [ ]:
print("── PASS 2: LOAD DATA ──")
start = time()

rows = []
global_index = 0

for f_id, file in enumerate(csv_files):
    print(f"[{f_id+1}/{len(csv_files)}] Load:", os.path.basename(file))

    for chunk in pd.read_csv(
        file,
        usecols=lambda c: c in FEATURES + [TARGET, 'Tile_ID'],
        chunksize=CHUNK_SIZE
    ):
        idx = range(global_index, global_index + len(chunk))

        mask = [i in final_set for i in idx]

        if any(mask):
            rows.append(chunk.loc[mask])

        global_index += len(chunk)

df_balanced = pd.concat(rows, ignore_index=True)

del rows
gc.collect()

print("✔ df_balanced:", df_balanced.shape)
print("⏱ Thời gian PASS 2:", time() - start)
print("──────────────────────────────\n")


── PASS 2: LOAD DATA ──
[1/9] Load: S2_145_TRAINING_data.csv
[2/9] Load: S2_149_TRAINING_data.csv
[3/9] Load: S2_157_TRAINING_data.csv
[4/9] Load: S2_105_TRAINING_data.csv
[5/9] Load: S1_237_TRAINING_data.csv
[6/9] Load: S1_49_TRAINING_data.csv
[7/9] Load: S1_9_TRAINING_data.csv
[8/9] Load: S1_274_TRAINING_data.csv
[9/9] Load: S_117_TRAINING_data.csv
✔ df_balanced: (200000, 25)
⏱ Thời gian PASS 2: 133.26376104354858
──────────────────────────────



In [ ]:

for c in df_balanced.columns:
    if c == TARGET:
        df_balanced[c] = df_balanced[c].astype(np.int8)
    else:
        df_balanced[c] = df_balanced[c].astype(np.float32)

print("RAM (MB):", df_balanced.memory_usage(deep=True).sum() / 1024**2)

OUT_PARQUET = "/content/drive/MyDrive/S2_TRAINING_BALANCED_200K.parquet"
df_balanced.to_parquet(OUT_PARQUET, index=False)

print("✔ Saved:", OUT_PARQUET)

RAM (MB): 18.501407623291016
✔ Saved: /content/drive/MyDrive/S2_TRAINING_BALANCED_200K.parquet


In [ ]:
import numpy as np
import pandas as pd
import gc
import joblib
import random

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
FEATURES = [
    'B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B11','B12',
    'NDVI','NDSI','NDWI','NDMI',
    'B11_B08_Ratio','B02_B04_Ratio',
    'B8A_B11_Ratio','B01_B11_Ratio',
    'B05_B04_Ratio','B12_B03_Ratio',
    'Tile_Cloud_Coverage'
]

TARGET = 'LABEL'


In [ ]:
X = df_balanced[FEATURES].astype(np.float32)
y = df_balanced[TARGET].astype(np.int8)

del df_balanced
gc.collect()

print("X:", X.shape, "y:", y.shape)

X: (200000, 23) y: (200000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
len(X_train)

160000

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score
import joblib, gc, os

def train_and_eval(model_name, model, params):
    print(f"\n===== TRAIN {model_name} =====")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=6,
        cv=2,
        scoring="f1_macro",
        n_jobs=1,
        random_state=42,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_model  = search.best_estimator_
    best_params = search.best_params_

    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="macro")

    print("Accuracy :", acc)
    print("F1 macro :", f1)
    print("Best params:", best_params)

    os.makedirs(OUTPUT_PATH, exist_ok=True)
    save_path = f"{OUTPUT_PATH}/{model_name}_best.pkl"
    joblib.dump(best_model, save_path)
    print("✔ Saved:", save_path)

    del search
    gc.collect()

    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1": f1,
        "Params": best_params
    }


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        random_state=42,
        max_iter=30000,
        dual=False,
        class_weight="balanced"
    ))
])

svm_params = {
    "svm__C": np.logspace(-3, 3, 20),
    "svm__tol": np.logspace(-5, -2, 6),
    "svm__loss": ["squared_hinge"]
}
svm_result = train_and_eval(
    "LinearSVM",
    svm_model,
    svm_params
)



===== TRAIN LinearSVM =====
Fitting 2 folds for each of 6 candidates, totalling 12 fits
Accuracy : 0.85805
F1 macro : 0.8578890846130117
Best params: {'svm__tol': np.float64(0.0025118864315095794), 'svm__loss': 'squared_hinge', 'svm__C': np.float64(1.438449888287663)}
✔ Saved: /content/drive/MyDrive/Models/LinearSVM_best.pkl


In [ ]:
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_params = {
    "n_estimators": [100, 150, 200],
    "max_depth": [10, 15, None],
    "max_features": ["sqrt"]
}

rf_result = train_and_eval(
    "RandomForest",
    rf_model,
    rf_params
)



===== TRAIN RandomForest =====
Fitting 2 folds for each of 6 candidates, totalling 12 fits
Accuracy : 0.91185
F1 macro : 0.9118478400517008
Best params: {'n_estimators': 200, 'max_features': 'sqrt', 'max_depth': None}
✔ Saved: /content/drive/MyDrive/Models/RandomForest_best.pkl


UnboundLocalError: cannot access local variable 'search' where it is not associated with a value

In [ ]:
lgb_model = lgb.LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=-1
)

lgb_params = {
    "n_estimators": [100, 150, 200],
    "num_leaves": [31, 50],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

lgb_result = train_and_eval(
    "LightGBM",
    lgb_model,
    lgb_params
)


===== TRAIN LightGBM =====
Fitting 2 folds for each of 6 candidates, totalling 12 fits
[LightGBM] [Info] Number of positive: 40000, number of negative: 40000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.087635 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5861
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 40000, number of negative: 40000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5861
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positi

In [ ]:
cat_model = CatBoostClassifier(
    loss_function="Logloss",
    random_state=42,
    verbose=0,
    thread_count=-1
)

cat_params = {
    "iterations": [100, 150, 200],
    "depth": [6, 8],
    "learning_rate": [0.05, 0.1]
}

cat_result = train_and_eval(
    "CatBoost",
    cat_model,
    cat_params
)



===== TRAIN CatBoost =====
Fitting 2 folds for each of 6 candidates, totalling 12 fits
Accuracy : 0.91125
F1 macro : 0.9112472167127161
Best params: {'learning_rate': 0.1, 'iterations': 150, 'depth': 8}
✔ Saved: /content/drive/MyDrive/Models/CatBoost_best.pkl


In [ ]:
xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_params = {
    "n_estimators": [100, 150, 200],
    "max_depth": [6, 8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

xgb_result = train_and_eval(
    "XGBoost",
    xgb_model,
    xgb_params
)



===== TRAIN XGBoost =====
Fitting 2 folds for each of 6 candidates, totalling 12 fits
Accuracy : 0.92165
F1 macro : 0.9216499951031247
Best params: {'subsample': 0.8, 'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
✔ Saved: /content/drive/MyDrive/Models/XGBoost_best.pkl


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/Models"

lgb_model = joblib.load(os.path.join(MODEL_DIR, "LightGBM_best.pkl"))
rf_model  = joblib.load(os.path.join(MODEL_DIR, "RandomForest_best.pkl"))
xgb_model = joblib.load(os.path.join(MODEL_DIR, "XGBoost_best.pkl"))
cat_model = joblib.load(os.path.join(MODEL_DIR, "CatBoost_best.pkl"))
svm_model = joblib.load(os.path.join(MODEL_DIR, "LinearSVM_best.pkl"))

print("✔ Loaded all base models")

✔ Loaded all base models


In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ("lgb", lgb_model),
        ("rf",  rf_model),
        ("xgb", xgb_model),
        ("cat", cat_model)
    ],
    voting="soft",
    n_jobs=1
)


In [ ]:
print("\n===== TRAIN VOTING ENSEMBLE =====")

voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average="macro")

print("Accuracy :", acc)
print("F1 macro :", f1)



===== TRAIN VOTING ENSEMBLE =====
[LightGBM] [Info] Number of positive: 80000, number of negative: 80000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5859
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Accuracy : 0.91905
F1 macro : 0.9190482496207775


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
import pandas as pd
import numpy as np

BIG_FILE = "/content/drive/MyDrive/S2_TRAINING_DATA_UNBALANCED_20_FEATURES (1).csv"
CHUNK_SIZE = 500_000

In [ ]:
def evaluate_on_big_data(model, model_name):
    print(f"\n=== EVALUATE {model_name} ON FULL DATA ===")

    y_true_all = []
    y_pred_all = []


    cols_in_file = pd.read_csv(BIG_FILE, nrows=1).columns.tolist()

    valid_features = [c for c in FEATURES if c in cols_in_file]

    missing = set(FEATURES) - set(valid_features)
    if missing:
        print("Missing features ignored:", missing)

    for chunk in pd.read_csv(
        BIG_FILE,
        usecols=valid_features + [TARGET],
        chunksize=CHUNK_SIZE
    ):
        X_chunk = chunk[valid_features]
        y_chunk = chunk[TARGET]

        y_pred = model.predict(X_chunk)

        y_true_all.append(y_chunk.values)
        y_pred_all.append(y_pred)

        del chunk, X_chunk, y_chunk

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    acc  = accuracy_score(y_true_all, y_pred_all)
    prec = precision_score(y_true_all, y_pred_all)
    rec  = recall_score(y_true_all, y_pred_all)
    f1   = f1_score(y_true_all, y_pred_all)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    }


In [ ]:
import joblib
import os

models = {
    "LightGBM": joblib.load("/content/drive/MyDrive/Models/LightGBM_best.pkl"),
    "XGBoost" : joblib.load("/content/drive/MyDrive/Models/XGBoost_best.pkl"),
    "CatBoost": joblib.load("/content/drive/MyDrive/Models/CatBoost_best.pkl"),
    "RF"      : joblib.load("/content/drive/MyDrive/Models/RandomForest_best.pkl"),
    "SVM"     : joblib.load("/content/drive/MyDrive/Models/LinearSVM_best.pkl")
}

print("✔ Models loaded")

✔ Models loaded


In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from time import time

CHUNK_SIZE = 500_000
SAMPLE_RATIO = 0.10


In [ ]:
print("── PASS 1: COUNT TOTAL ROWS ──")
start = time()

TOTAL_ROWS = 0

for f_id, file in enumerate(csv_files):
    print(f"[{f_id+1}/{len(csv_files)}] Count:", os.path.basename(file))

    for chunk in pd.read_csv(
        file,
        usecols=[TARGET],
        chunksize=CHUNK_SIZE
    ):
        TOTAL_ROWS += len(chunk)

print("✔ TOTAL ROWS:", TOTAL_ROWS)
print("Time:", time() - start)


── PASS 1: COUNT TOTAL ROWS ──
[1/9] Count: S2_145_TRAINING_data.csv
[2/9] Count: S2_149_TRAINING_data.csv
[3/9] Count: S2_157_TRAINING_data.csv
[4/9] Count: S2_105_TRAINING_data.csv
[5/9] Count: S1_237_TRAINING_data.csv
[6/9] Count: S1_49_TRAINING_data.csv
[7/9] Count: S1_9_TRAINING_data.csv
[8/9] Count: S1_274_TRAINING_data.csv
[9/9] Count: S_117_TRAINING_data.csv
✔ TOTAL ROWS: 25437467
Time: 88.68627381324768


In [ ]:
MAX_ROWS = int(TOTAL_ROWS * SAMPLE_RATIO)

print("✔ TAKE FIRST ROWS:", MAX_ROWS)


✔ TAKE FIRST ROWS: 2543746


In [ ]:
SAMPLES_PER_FILE = 200_000
rng = np.random.default_rng(42)

selected_pos = set()   # (file_id, local_index)

for f_id, file in enumerate(csv_files):
    print(f"[{f_id+1}/{len(csv_files)}] Sample:", os.path.basename(file))

    # ---- đếm số dòng trong file ----
    total_rows = 0
    for chunk in pd.read_csv(
        file,
        usecols=[TARGET],
        chunksize=CHUNK_SIZE
    ):
        total_rows += len(chunk)

    if total_rows == 0:
        continue

    # ---- mỗi file lấy 200k (hoặc ít hơn nếu file nhỏ) ----
    k = min(SAMPLES_PER_FILE, total_rows)

    # ---- chọn ngẫu nhiên local index ----
    sampled_local_idx = rng.choice(
        total_rows,
        size=k,
        replace=False
    )

    for idx in sampled_local_idx:
        selected_pos.add((f_id, idx))

    print(f"    -> sampled {k} / {total_rows}")

print("✔ Total sampled:", len(selected_pos))


[1/9] Sample: S2_145_TRAINING_data.csv
    -> sampled 200000 / 2570000
[2/9] Sample: S2_149_TRAINING_data.csv
    -> sampled 200000 / 2079424
[3/9] Sample: S2_157_TRAINING_data.csv
    -> sampled 200000 / 2570000
[4/9] Sample: S2_105_TRAINING_data.csv
    -> sampled 200000 / 1308043
[5/9] Sample: S1_237_TRAINING_data.csv
    -> sampled 200000 / 2060000
[6/9] Sample: S1_49_TRAINING_data.csv
    -> sampled 200000 / 2560000
[7/9] Sample: S1_9_TRAINING_data.csv
    -> sampled 200000 / 2450000
[8/9] Sample: S1_274_TRAINING_data.csv
    -> sampled 200000 / 2470000
[9/9] Sample: S_117_TRAINING_data.csv
    -> sampled 200000 / 7370000
✔ Total sampled: 1800000


In [ ]:
print("── PASS 3: LOAD DATA BY (file_id, local_index) ──")
start = time()

rows = []

for f_id, file in enumerate(csv_files):
    print(f"[{f_id+1}/{len(csv_files)}] Load:", os.path.basename(file))

    local_index = 0

    for chunk in pd.read_csv(
        file,
        usecols=FEATURES + [TARGET],
        chunksize=CHUNK_SIZE
    ):
        n = len(chunk)

        idx = np.arange(local_index, local_index + n)

        mask = [
            (f_id, i) in selected_pos
            for i in idx
        ]

        if any(mask):
            rows.append(chunk.loc[mask])

        local_index += n

df_10percent = pd.concat(rows, ignore_index=True)

print("✔ df_10percent shape:", df_10percent.shape)
print("Time:", time() - start)


── PASS 3: LOAD DATA BY (file_id, local_index) ──
[1/9] Load: S2_145_TRAINING_data.csv
[2/9] Load: S2_149_TRAINING_data.csv
[3/9] Load: S2_157_TRAINING_data.csv
[4/9] Load: S2_105_TRAINING_data.csv
[5/9] Load: S1_237_TRAINING_data.csv
[6/9] Load: S1_49_TRAINING_data.csv
[7/9] Load: S1_9_TRAINING_data.csv
[8/9] Load: S1_274_TRAINING_data.csv
[9/9] Load: S_117_TRAINING_data.csv
✔ df_10percent shape: (1800000, 24)
Time: 153.90852284431458


In [ ]:
X_eval = df_10percent[FEATURES]
y_eval = df_10percent[TARGET]

print("Eval shape:", X_eval.shape)

Eval shape: (1800000, 23)


In [ ]:
import joblib

models = {
    "RandomForest":      joblib.load("/content/drive/MyDrive/Models/RandomForest_best.pkl"),
    "CatBoost":      joblib.load("/content/drive/MyDrive/Models/CatBoost_best.pkl"),
    "LightGBM":      joblib.load("/content/drive/MyDrive/Models/LightGBM_best.pkl"),
    "XGBoost":      joblib.load("/content/drive/MyDrive/Models/XGBoost_best.pkl")
}


In [ ]:
FEATURES = [
    'B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B11','B12',
    'NDVI','NDSI','NDWI','NDMI',
    'B11_B08_Ratio','B02_B04_Ratio',
    'B8A_B11_Ratio','B01_B11_Ratio',
    'B05_B04_Ratio','B12_B03_Ratio',
    'Tile_Cloud_Coverage'
]

In [ ]:

def evaluate_model(model, X, y):
    X = X[FEATURES]

    y_pred = model.predict(X)

    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec  = recall_score(y, y_pred)
    f1   = f1_score(y, y_pred)

    return acc, prec, rec, f1

In [ ]:
results = []

for name, model in models.items():
    print(f"\n=== EVALUATE {name} ===")

    acc, prec, rec, f1 = evaluate_model(model, X_eval, y_eval)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })



=== EVALUATE RandomForest ===
Accuracy : 0.8944
Precision: 0.9670
Recall   : 0.8481
F1-score : 0.9037

=== EVALUATE LightGBM ===
Accuracy : 0.9130
Precision: 0.9584
Recall   : 0.8895
F1-score : 0.9227

=== EVALUATE CatBoost ===


AttributeError: 'CatBoostClassifier' object has no attribute 'feature_names_in_'

In [ ]:
results = []

for name, model in models.items():
    print(f"\n=== EVALUATE {name} ===")

    acc, prec, rec, f1 = evaluate_model(model, X_eval, y_eval)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })



=== EVALUATE CatBoost ===
Accuracy : 0.9082
Precision: 0.9582
Recall   : 0.8811
F1-score : 0.9180

=== EVALUATE XGBoost ===


ValueError: feature_names mismatch: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'NDVI', 'NDSI', 'NDWI', 'NDMI', 'B11_B08_Ratio', 'B02_B04_Ratio', 'B8A_B11_Ratio', 'B01_B11_Ratio', 'B05_B04_Ratio', 'B12_B03_Ratio', 'Tile_Cloud_Coverage'] ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'NDVI', 'NDSI', 'NDWI', 'B11_B08_Ratio', 'B02_B04_Ratio', 'NDMI', 'B8A_B11_Ratio', 'B01_B11_Ratio', 'B05_B04_Ratio', 'B12_B03_Ratio', 'Tile_Cloud_Coverage']

In [ ]:
results = []

for name, model in models.items():
    print(f"\n=== EVALUATE {name} ===")

    acc, prec, rec, f1 = evaluate_model(model, X_eval, y_eval)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })



=== EVALUATE XGBoost ===
Accuracy : 0.9182
Precision: 0.9585
Recall   : 0.8988
F1-score : 0.9277


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\n===== TEST VOTING ENSEMBLE =====")

# ép đúng thứ tự feature
X_test_eval = X_test[FEATURES]

# Predict
y_pred = voting_clf.predict(X_test_eval)

# Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")


===== TEST VOTING ENSEMBLE =====
Accuracy : 0.9193
Precision: 0.9202
Recall   : 0.9183
F1-score : 0.9192


In [ ]:
results = []

for name, model in models.items():
    print(f"\n=== EVALUATE {name} ===")

    acc, prec, rec, f1 = evaluate_model(model, X_eval, y_eval)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })



=== EVALUATE SVM ===
Accuracy : 0.8553
Precision: 0.9083
Recall   : 0.8187
F1-score : 0.8612


In [ ]:
results = []

for name, model in models.items():
    print(f"\n=== EVALUATE {name} ===")

    acc, prec, rec, f1 = evaluate_model(model, X_eval, y_eval)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })



=== EVALUATE RandomForest ===
Accuracy : 0.9164
Precision: 0.9399
Recall   : 0.9055
F1-score : 0.9224

=== EVALUATE CatBoost ===
Accuracy : 0.9164
Precision: 0.9381
Recall   : 0.9074
F1-score : 0.9225

=== EVALUATE LightGBM ===
Accuracy : 0.9250
Precision: 0.9434
Recall   : 0.9183
F1-score : 0.9307

=== EVALUATE XGBoost ===
Accuracy : 0.9269
Precision: 0.9436
Recall   : 0.9218
F1-score : 0.9325


In [ ]:
results = []

for name, model in models.items():
    print(f"\n=== EVALUATE {name} ===")

    acc, prec, rec, f1 = evaluate_model(model, X_eval, y_eval)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })



=== EVALUATE CatBoost ===
Accuracy : 0.9163
Precision: 0.9385
Recall   : 0.9070
F1-score : 0.9224

=== EVALUATE LightGBM ===
Accuracy : 0.9249
Precision: 0.9439
Recall   : 0.9176
F1-score : 0.9306

=== EVALUATE XGBoost ===
Accuracy : 0.9266
Precision: 0.9436
Recall   : 0.9212
F1-score : 0.9323


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\n===== TEST VOTING ENSEMBLE =====")

# ép đúng thứ tự feature
X_test_eval = X_test[FEATURES]

# Predict
y_pred = voting_clf.predict(X_test_eval)

# Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")


===== TEST VOTING ENSEMBLE =====
Accuracy : 0.9191
Precision: 0.9230
Recall   : 0.9144
F1-score : 0.9187


In [ ]:
import joblib

model_path = os.path.join(MODEL_DIR, "voting_ensemble.pkl")

joblib.dump(voting_clf, model_path)

print("✔ Voting model saved to:", model_path)

✔ Voting model saved to: /content/drive/MyDrive/Models/voting_ensemble.pkl


In [ ]:
!pip install pytorch-tabnet

In [ ]:
import numpy as np
import torch

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
from sklearn.model_selection import train_test_split

# Tách validation từ train (10–20% là hợp lý)
X_train_df, X_val_df, y_train_ser, y_val_ser = train_test_split(
    X_train[FEATURES],
    y_train,
    test_size=0.15,          # 15% làm validation
    random_state=42,
    stratify=y_train         # QUAN TRỌNG cho F1
)

# Chuyển sang numpy cho TabNet
X_train_tab = X_train_df.values
X_val_tab   = X_val_df.values
X_test_tab  = X_test[FEATURES].values

y_train_tab = y_train_ser.values
y_val_tab   = y_val_ser.values
y_test_tab  = y_test.values


In [ ]:
tabnet = TabNetClassifier(
    n_d=32,                # dimension decision
    n_a=32,                # dimension attention
    n_steps=5,             # số step attention
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    lambda_sparse=1e-4,

    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),

    mask_type="entmax",    # tốt hơn sparsemax
    scheduler_params={"step_size":20, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,

    verbose=10,
    seed=42
)


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [ ]:
tabnet.fit(
    X_train=X_train_tab,
    y_train=y_train_tab,

    eval_set=[(X_val_tab, y_val_tab)],
    eval_name=["val"],
    eval_metric=["accuracy"],

    max_epochs=200,
    patience=20,
    batch_size=8192,
    virtual_batch_size=1024,
    num_workers=0,
    drop_last=False
)


epoch 0  | loss: 0.48263 | val_accuracy: 0.59679 |  0:00:21s
epoch 10 | loss: 0.25403 | val_accuracy: 0.67171 |  0:02:47s
epoch 20 | loss: 0.2408  | val_accuracy: 0.87162 |  0:05:17s
epoch 30 | loss: 0.22879 | val_accuracy: 0.89917 |  0:07:44s
epoch 40 | loss: 0.2219  | val_accuracy: 0.899   |  0:10:10s
epoch 50 | loss: 0.22034 | val_accuracy: 0.89879 |  0:12:36s
epoch 60 | loss: 0.21169 | val_accuracy: 0.90483 |  0:15:04s
epoch 70 | loss: 0.20818 | val_accuracy: 0.90396 |  0:17:30s
epoch 80 | loss: 0.20661 | val_accuracy: 0.90662 |  0:19:55s
epoch 90 | loss: 0.19939 | val_accuracy: 0.90758 |  0:22:22s
epoch 100| loss: 0.19649 | val_accuracy: 0.90558 |  0:24:46s
epoch 110| loss: 0.1982  | val_accuracy: 0.90558 |  0:27:11s
epoch 120| loss: 0.18997 | val_accuracy: 0.90633 |  0:29:36s
epoch 130| loss: 0.18574 | val_accuracy: 0.90504 |  0:32:01s

Early stopping occurred at epoch 133 with best_epoch = 113 and best_val_accuracy = 0.90821


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [ ]:
y_pred = tabnet.predict(X_test_tab)

acc  = accuracy_score(y_test_tab, y_pred)
prec = precision_score(y_test_tab, y_pred)
rec  = recall_score(y_test_tab, y_pred)
f1   = f1_score(y_test_tab, y_pred)

print("===== TABNET TEST =====")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

===== TABNET TEST =====
Accuracy : 0.9087
Precision: 0.9123
Recall   : 0.9044
F1-score : 0.9083


In [ ]:
X_eval = df_10percent[FEATURES].values
y_eval = df_10percent[TARGET].values

print("Eval shape:", X_eval.shape)

Eval shape: (1800000, 23)


In [ ]:
y_pred = tabnet.predict(X_eval)
y_prob = tabnet.predict_proba(X_eval)[:, 1]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc  = accuracy_score(y_eval, y_pred)
prec = precision_score(y_eval, y_pred)
rec  = recall_score(y_eval, y_pred)
f1   = f1_score(y_eval, y_pred)

print("===== TABNET EVAL on df_10percent =====")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")


===== TABNET EVAL on df_10percent =====
Accuracy : 0.9152
Precision: 0.9352
Recall   : 0.9083
F1-score : 0.9216
